<a href="https://colab.research.google.com/github/macaluzate/Aprendizaje_Auto_Proyect/blob/main/EntrenamientoMalwarePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dotenv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from dotenv import load_dotenv
import os
import zipfile
import pandas as pd
from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [ ]:
# ----------------- Función principal para obtener el DataFrame procesado -----------------

# Función para descargar el dataset de Kaggle y descomprimirlo en directorio de trabajo
def download_dataset():
    load_dotenv()
    os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = os.getenv("KAGGLE_KEY")

    from kaggle.api.kaggle_api_extended import KaggleApi

    print("Autenticando en Kaggle...")
    # Inicializamos y autenticamos la API
    api = KaggleApi()
    api.authenticate()

    print("Descargando el dataset de la competencia 'microsoft-malware-prediction'...")
    # Descargamos los archivos de la competencia
    api.competition_download_files(
        competition="microsoft-malware-prediction",
        quiet=False
    )

    print("Descomprimiendo 'train.csv' del archivo ZIP descargado...")
    # Descomprimir el ZIP descargado
    with zipfile.ZipFile("microsoft-malware-prediction.zip", "r") as zip_ref:
        for member in zip_ref.namelist():
            if member.endswith("train.csv"):
                zip_ref.extract(member)
                break

    # Borrar el ZIP para liberar espacio
    print("Borrando el archivo ZIP descargado para ahorrar almacenamiento...")
    os.remove("microsoft-malware-prediction.zip")
    print("Dataset descargado y descomprimido correctamente.")



# Función para leer el archivo CSV y devolver un DataFrame de pandas
# Esta función permite especificar tipos de datos personalizados y columnas a leer.
def read_malware_csv(file_path = 'train.csv', dtypes_dict=None, columns=None):
    """
    Lee un archivo CSV y devuelve un DataFrame de pandas.

    Args:
        file_path: Ruta al archivo CSV.

    Returns:
        Un DataFrame de pandas con los datos del archivo CSV.
    """

    print(f"Leyendo el archivo CSV desde {file_path}...")
    if dtypes_dict is not None:
        print("Usando tipos de datos personalizados para las columnas...")
        train_df = pd.read_csv(file_path, dtype=dtypes_dict, usecols=columns)
    else:
        train_df = pd.read_csv(file_path, usecols=columns)

    return train_df

In [ ]:
import pandas as pd
import numpy as np
import re

# ------------------------ Funciones utilitarias ------------------------

def llenar_nulos_con_texto(df, columnas, texto="UNKNOWN"):
    for col in columnas:
        if col in df.columns:
            df[col] = df[col].fillna(texto)

def llenarNulos(train_df, columnas):
    for col in columnas:
        if col in train_df.columns:
            if pd.api.types.is_numeric_dtype(train_df[col]):
                train_df[col] = train_df[col].fillna(-1)
            else:
                train_df[col] = train_df[col].fillna('Missing')


def imputar_con_media_y_marcar_nulos(df, columnas):
    for col in columnas:
        if col in df.columns:
            df[f'{col}_missing'] = df[col].isna().astype(int)
            # df[col].fillna(df[col].mean(), inplace=True)
            df[col] = df[col].fillna(df[col].mean())


# ------------------------ Agrupación y limpieza ------------------------

def limpiar_smartscreen(train_df):
    if 'SmartScreen' in train_df.columns:
        train_df['SmartScreen'] = (
            train_df['SmartScreen']
            .str.lower()
            .replace({
                'enabled': 'on', 'requireadmin': 'requireadmin',
                'promt': 'prompt', 'promprt': 'prompt', 'prompt ': 'prompt',
                '0': 'off', '00000000': 'off'
            })
            .fillna('NaNNN')
        )

def limpiar_power_platform(train_df):
    if 'Census_PowerPlatformRoleName' in train_df.columns:
        train_df['Census_PowerPlatformRoleName'] = train_df['Census_PowerPlatformRoleName'].replace('NaN', 'UNKNOWN').fillna('UNKNOWN')
    if 'Census_ChassisTypeName' in train_df.columns:
        valores_permitidos = ['Notebook', 'Desktop', 'Laptop', 'UNKNOWN', 'nan', 'Unknown']
        train_df['Census_ChassisTypeName'] = train_df['Census_ChassisTypeName'].fillna('UNKNOWN').apply(
            lambda x: x if x in valores_permitidos else 'Other')

def agrupar_valores_poco_representativos(train_df, columna, umbral=0.02, nombre_categoria='Others'):
    if columna in train_df.columns:
        frecuencias = train_df[columna].value_counts(normalize=True)
        categorias_poco_frecuentes = frecuencias[frecuencias < umbral].index
        train_df[columna] = train_df[columna].apply(lambda x: nombre_categoria if x in categorias_poco_frecuentes else x)

def agrupar_rama(rama):
    if isinstance(rama, str):
        if re.match(r'^rs[0-9]+', rama):
            return re.match(r'^rs[0-9]+', rama).group(0)
        elif rama.startswith('rs'):
            return 'rs'
        elif re.match(r'^th[0-9]+', rama):
            return re.match(r'^th[0-9]+', rama).group(0)
        elif rama.startswith('th'):
            return 'th'
        elif rama.startswith('win'):
            return 'win'
    return 'khmer'

def agrupar_canal(canal):
    if isinstance(canal, str):
        if canal.startswith('OEM'):
            return 'OEM'
        elif canal.startswith('Retail'):
            return 'Retail'
    return 'Volume'



In [ ]:
def preprocess_data(train_df):
  llenarNull = ['RtpStateBitfield', 'DefaultBrowsersIdentifier', 'CityIdentifier', 'GeoNameIdentifier','OrganizationIdentifier', 'OsBuildLab', 'SMode',
          'Firewall',
          'UacLuaenable',
          'Census_TotalPhysicalRAM',
          'Census_InternalPrimaryDiagonalDisplaySizeInInches',
          'Census_InternalPrimaryDisplayResolutionVertical',
          'Census_InternalPrimaryDisplayResolutionHorizontal',
          'Census_IsFlightingInternal', 'Census_FirmwareManufacturerIdentifier',
          'Census_IsWIMBootEnabled', 'Census_IsVirtualDevice',
          'Census_IsAlwaysOnAlwaysConnectedCapable',
          'Wdft_IsGamer', 'Census_IsFlightingInternal','Census_FirmwareManufacturerIdentifier', 'Census_IsWIMBootEnabled']

  columnas_nulos_texto = ['Census_PrimaryDiskTypeName']
  # Aplicar limpieza modular usando las funciones importadas
  llenar_nulos_con_texto(train_df, columnas_nulos_texto, "UNKNOWN")

  llenarNulos(train_df, llenarNull)
  limpiar_smartscreen(train_df)
  agrupar_valores_poco_representativos(train_df, 'SmartScreen')
  # train_df['SmartScreen'] = agrupar_valores_poco_representativos(train_df, 'SmartScreen', umbral=0.02, nombre_categoria='Others')
  # df = limpiar_chassis_type(df)
  limpiar_power_platform(train_df)

  # configs = categorizar(train_df, cat)

  # Otros procesamiento individuales
  train_df['Census_OSBranch'] = train_df['Census_OSBranch'].apply(agrupar_rama)
  train_df['Census_GenuineStateName'] = train_df['Census_GenuineStateName'].apply(lambda x: 0 if x == 'IS_GENUINE' else 1)
  train_df['Census_ActivationChannel'] = train_df['Census_ActivationChannel'].apply(agrupar_canal)

  # Columnas para aplicar One-Hot Encoding
  # columns_to_one_hot = []

  # Aplicamos One-Hot Encoding a las columnas especificadas
  # for col in columns_to_one_hot:
      # train_df = dummy_encode(train_df, col)

  imputar_con_media_y_marcar_nulos(train_df, ['AVProductsInstalled','AVProductsEnabled'])

  return train_df

In [ ]:
download_dataset()

Autenticando en Kaggle...
Descargando el dataset de la competencia 'microsoft-malware-prediction'...


100%|██████████| 1.54G/1.54G [00:07<00:00, 213MB/s]



Descomprimiendo 'train.csv' del archivo ZIP descargado...
Borrando el archivo ZIP descargado para ahorrar almacenamiento...
Dataset descargado y descomprimido correctamente.


In [ ]:
# Leamos el .json con el diccionario de los dtypes
with open('dtypes_dict.json', 'r') as f:
    dtypes_dict = json.load(f)

# Columnas a leer del CSV
columns_to_read = ['ProductName', 'EngineVersion', 'AppVersion', 'AvSigVersion',
            'IsBeta', 'RtpStateBitfield', 'IsSxsPassiveMode', 'DefaultBrowsersIdentifier',
            'AVProductsInstalled', 'AVProductsEnabled', 'HasTpm',
            'CountryIdentifier', 'CityIdentifier', 'OrganizationIdentifier', 'GeoNameIdentifier',
            'LocaleEnglishNameIdentifier', 'Platform', 'Processor', 'OsVer', 'OsBuild', 'OsSuite',
            'OsPlatformSubRelease','OsBuildLab', 'SkuEdition', 'IsProtected', 'AutoSampleOptIn',
            'SMode', 'SmartScreen', 'Firewall', 'UacLuaenable', 'Census_MDC2FormFactor', 'Census_DeviceFamily', 'Census_OEMNameIdentifier',
            'Census_OEMModelIdentifier', 'Census_ProcessorCoreCount', 'Census_ProcessorManufacturerIdentifier',
            'Census_PrimaryDiskTotalCapacity', 'Census_PrimaryDiskTypeName', 'Census_SystemVolumeTotalCapacity', 'Census_TotalPhysicalRAM',
            'Census_ChassisTypeName', 'Census_InternalPrimaryDiagonalDisplaySizeInInches', 'Census_InternalPrimaryDisplayResolutionHorizontal',
            'Census_InternalPrimaryDisplayResolutionVertical', 'Census_PowerPlatformRoleName',
            'Census_OSArchitecture', 'Census_OSBranch', 'Census_OSWUAutoUpdateOptionsName',
            'Census_IsPortableOperatingSystem', 'Census_GenuineStateName',
            'Census_ActivationChannel', 'Census_IsFlightingInternal',
            'Census_FirmwareManufacturerIdentifier', 'Census_IsSecureBootEnabled',
            'Census_IsWIMBootEnabled', 'Census_IsVirtualDevice', 'Census_IsTouchEnabled',
            'Census_IsPenCapable', 'Census_IsAlwaysOnAlwaysConnectedCapable','Wdft_IsGamer',
            # Variable objetivo
            'HasDetections'
]

In [ ]:
# Leemos el archivo CSV para obtener el DataFrame de pandas
df = read_malware_csv(file_path='train.csv', dtypes_dict=dtypes_dict,
                          columns=columns_to_read)

Leyendo el archivo CSV desde train.csv...
Usando tipos de datos personalizados para las columnas...


In [ ]:
df.head()

,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductsInstalled,AVProductsEnabled,...,Census_IsFlightingInternal,Census_FirmwareManufacturerIdentifier,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,HasDetections
0,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1735.0,0,7.0,0,NaN,1.0,1.0,...,NaN,628.0,0,NaN,0.0,0,0,0.0,0.0,0
1,win8defender,1.1.14600.4,4.13.17134.1,1.263.48.0,0,7.0,0,NaN,1.0,1.0,...,NaN,628.0,0,NaN,0.0,0,0,0.0,0.0,0
2,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1341.0,0,7.0,0,NaN,1.0,1.0,...,NaN,142.0,0,NaN,0.0,0,0,0.0,0.0,0
3,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1527.0,0,7.0,0,NaN,1.0,1.0,...,NaN,355.0,0,NaN,0.0,0,0,0.0,0.0,1
4,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1379.0,0,7.0,0,NaN,1.0,1.0,...,0.0,355.0,0,0.0,0.0,0,0,0.0,0.0,1


In [ ]:
from sklearn.model_selection import train_test_split

# Dividimos en 80% entrenamiento y 20% prueba, manteniendo HasDetections
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df["HasDetections"]
)

In [ ]:
train_df_proc = preprocess_data(train_df)

In [ ]:
train_df_proc.head()

,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductsInstalled,AVProductsEnabled,...,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,HasDetections,AVProductsInstalled_missing,AVProductsEnabled_missing
7991317,win8defender,1.1.15200.1,4.18.1807.18075,1.275.511.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,1.0,1,0,0
6442680,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1228.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,1.0,0,0,0
75622,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1863.0,0,7.0,0,-1.0,1.0,1.0,...,0,0.0,0.0,0,0,0.0,1.0,1,0,0
5857364,win8defender,1.1.15200.1,4.13.17134.228,1.275.1011.0,0,7.0,0,-1.0,3.0,1.0,...,1,-1.0,0.0,0,0,0.0,1.0,1,0,0
3078999,win8defender,1.1.15200.1,4.18.1807.18075,1.275.72.0,0,7.0,0,-1.0,1.0,1.0,...,1,-1.0,0.0,0,0,0.0,1.0,0,0,0


In [ ]:
test_df_proc = preprocess_data(test_df)

In [ ]:
test_df_proc.head()

,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductsInstalled,AVProductsEnabled,...,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,HasDetections,AVProductsInstalled_missing,AVProductsEnabled_missing
3965549,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1668.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,0.0,1,0,0
6676837,win8defender,1.1.15200.1,4.18.1807.18075,1.275.1378.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,0.0,0,0,0
3947217,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1073.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,0.0,0,0,0
255770,win8defender,1.1.15100.1,4.18.1806.18062,1.273.488.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,0.0,1,0,0
5485372,win8defender,1.1.15300.6,4.13.17134.228,1.277.4.0,0,7.0,0,-1.0,1.0,1.0,...,0,-1.0,0.0,0,0,0.0,0.0,0,0,0


# Baseline
Con el objetivo de establecer un punto de referencia o baseline sobre el cual evaluar mejoras posteriores, se entrenaron tres modelos representativos que cubren distintos niveles de complejidad y enfoques en aprendizaje automático. Cabe resaltar que cada modelo fue entrenado sobre el mismo conjunto de datos preprocesado, y sus resultados fueron comparados utilizando algunas métricas estándar como accuracy, AUC (Área Bajo la Curva ROC) y F1-score, con el fin de identificar cuál presenta el mejor comportamiento general como línea base.

A continuación, se presentan los resultados individuales obtenidos por cada uno de estos modelos:

## LightGBM

LightGBM es un modelo de boosting de gradiente que optimiza el rendimiento con alta eficiencia computacional y ha mostrado excelentes resultados en tareas sobre datos tabulares.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from lightgbm import LGBMClassifier

# 1. Definimos las columnas para hacer OneHotEncoding y TargetEncoding
onehot_cols = ['Census_OSArchitecture', 'Census_OSBranch',
                        'Census_OSWUAutoUpdateOptionsName',
                        'Census_ActivationChannel', 'ProductName',
                        'SkuEdition','Census_PrimaryDiskTypeName',
                        'Census_PowerPlatformRoleName', 'ProductName',
                        'Platform','Processor','OsPlatformSubRelease',
                        'SkuEdition', 'Census_MDC2FormFactor',
                        'Census_DeviceFamily', 'Census_PrimaryDiskTypeName',
                        'Census_PowerPlatformRoleName','Census_OSArchitecture',
                        'Census_OSBranch','Census_OSWUAutoUpdateOptionsName',
                        'Census_ActivationChannel']
target_cols = ['EngineVersion','AppVersion', 'AvSigVersion','DefaultBrowsersIdentifier', 'CountryIdentifier', 'CityIdentifier', 'GeoNameIdentifier', 'OrganizationIdentifier', 'OsVer', 'OsBuild', 'OsSuite', 'OsBuildLab', 'Census_FirmwareManufacturerIdentifier', 'SmartScreen', 'Census_ChassisTypeName']

# 2. Dividimos las features y el target
X_train = train_df_proc.drop(columns='HasDetections')
y_train = train_df_proc['HasDetections']
X_test = test_df_proc.drop(columns='HasDetections')
y_test = test_df_proc['HasDetections']

# 3. Rellena los nulos antes
for col in onehot_cols + target_cols:
    X_train[col] = X_train[col].fillna('Missing')
    X_test[col] = X_test[col].fillna('Missing')


In [ ]:
X_test.head()

,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductsInstalled,AVProductsEnabled,...,Census_FirmwareManufacturerIdentifier,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,AVProductsInstalled_missing,AVProductsEnabled_missing
3965549,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1668.0,0,7.0,0,-1.0,1.0,1.0,...,355.0,0,-1.0,0.0,0,0,0.0,0.0,0,0
6676837,win8defender,1.1.15200.1,4.18.1807.18075,1.275.1378.0,0,7.0,0,-1.0,1.0,1.0,...,512.0,0,-1.0,0.0,0,0,0.0,0.0,0,0
3947217,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1073.0,0,7.0,0,-1.0,1.0,1.0,...,897.0,0,-1.0,0.0,0,0,0.0,0.0,0,0
255770,win8defender,1.1.15100.1,4.18.1806.18062,1.273.488.0,0,7.0,0,-1.0,1.0,1.0,...,628.0,0,-1.0,0.0,0,0,0.0,0.0,0,0
5485372,win8defender,1.1.15300.6,4.13.17134.228,1.277.4.0,0,7.0,0,-1.0,1.0,1.0,...,628.0,0,-1.0,0.0,0,0,0.0,0.0,0,0


In [ ]:
# 4. Preprocesamiento: combinación de OneHot y TargetEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
        #('target', TargetEncoder(cols=target_cols), target_cols)
        ('target', TargetEncoder(smooth="auto"), target_cols)
    ],
    remainder='passthrough'
)

# 5. Construímos pipeline con el modelo
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LGBMClassifier(random_state=42))
])

# 6. Entrenamos con X_train, y_train
pipeline.fit(X_train, y_train)

# 7. Realizamos predicciones sobre el conjunto de prueba y calculamos métricas de evaluación
y_pred = pipeline.predict(X_test) # Etiquetas predichas
y_proba = pipeline.predict_proba(X_test)[:, 1] # Probabilidades para la clase positiva (1)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 3567113, number of negative: 3570073
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.941476 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4323
[LightGBM] [Info] Number of data points in the train set: 7137186, number of used features: 164
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499793 -> initscore=-0.000829
[LightGBM] [Info] Start training from score -0.000829


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

print("AUC:", roc_auc_score(y_test, y_proba))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

AUC: 0.7079056650913024
Accuracy: 0.6471389012031069
F1 Score: 0.6409257777501493


## RandomForestClassifier

 Ejemplo de modelo de tipo ensemble basado en árboles, robusto frente a ruido y capaz de capturar relaciones no lineales.

In [ ]:
# 1. Definimos las columnas para OneHotEncoding y TargetEncoding
onehot_cols = ['Census_OSArchitecture', 'Census_OSBranch',
                        'Census_OSWUAutoUpdateOptionsName',
                        'Census_ActivationChannel', 'ProductName',
                        'SkuEdition','Census_PrimaryDiskTypeName',
                        'Census_PowerPlatformRoleName', 'ProductName',
                        'Platform','Processor','OsPlatformSubRelease',
                        'SkuEdition', 'Census_MDC2FormFactor',
                        'Census_DeviceFamily', 'Census_PrimaryDiskTypeName',
                        'Census_PowerPlatformRoleName','Census_OSArchitecture',
                        'Census_OSBranch','Census_OSWUAutoUpdateOptionsName',
                        'Census_ActivationChannel']
target_cols = ['EngineVersion','AppVersion', 'AvSigVersion','DefaultBrowsersIdentifier', 'CountryIdentifier', 'CityIdentifier', 'GeoNameIdentifier', 'OrganizationIdentifier', 'OsVer', 'OsBuild', 'OsSuite', 'OsBuildLab', 'Census_FirmwareManufacturerIdentifier', 'SmartScreen', 'Census_ChassisTypeName']

# 2. Dividimos features y target
X_train_random = train_df_proc.drop(columns='HasDetections')
y_train_random = train_df_proc['HasDetections']
X_test_random = test_df_proc.drop(columns='HasDetections')
y_test_random = test_df_proc['HasDetections']

# 3. Rellenamos nulos antes
for col in onehot_cols + target_cols:
    X_train_random[col] = X_train_random[col].fillna('Missing')
    X_test_random[col] = X_test_random[col].fillna('Missing')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

# Preprocesado
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
        ('target', TargetEncoder(smooth="auto"), target_cols)
    ],
    remainder='passthrough'
)

# Pipeline con Random Forest
pipeline_rf = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1,
        verbose=1  # Para ver progreso
    ))
])

# Entrenamos el modelo
pipeline_rf.fit(X_train_random, y_train_random)

# Evalúa
y_pred_rf = pipeline_rf.predict(X_test_random)
y_proba_rf = pipeline_rf.predict_proba(X_test_random)[:, 1]

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  6.1min finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    1.3s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    3.9s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    1.4s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    4.0s finished


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test_random, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test_random, y_proba_rf))

Accuracy: 0.6412508679889054
ROC AUC: 0.7007995343138567


## Logistic Regression

Método lineal simple, usado aquí para identificar si existe una separación lineal significativa entre clases.

In [ ]:
# 1. Defininmos las columnas para OneHotEncoding y TargetEncoding
onehot_cols = ['Census_OSArchitecture', 'Census_OSBranch',
                        'Census_OSWUAutoUpdateOptionsName',
                        'Census_ActivationChannel', 'ProductName',
                        'SkuEdition','Census_PrimaryDiskTypeName',
                        'Census_PowerPlatformRoleName', 'ProductName',
                        'Platform','Processor','OsPlatformSubRelease',
                        'SkuEdition', 'Census_MDC2FormFactor',
                        'Census_DeviceFamily', 'Census_PrimaryDiskTypeName',
                        'Census_PowerPlatformRoleName','Census_OSArchitecture',
                        'Census_OSBranch','Census_OSWUAutoUpdateOptionsName',
                        'Census_ActivationChannel']
target_cols = ['EngineVersion','AppVersion', 'AvSigVersion','DefaultBrowsersIdentifier', 'CountryIdentifier', 'CityIdentifier', 'GeoNameIdentifier', 'OrganizationIdentifier', 'OsVer', 'OsBuild', 'OsSuite', 'OsBuildLab', 'Census_FirmwareManufacturerIdentifier', 'SmartScreen', 'Census_ChassisTypeName']

# 2. Dividimos features y target
X_train_log = train_df_proc.drop(columns='HasDetections')
y_train_log = train_df_proc['HasDetections']
X_test_log = test_df_proc.drop(columns='HasDetections')
y_test_log = test_df_proc['HasDetections']

# 3. Rellenamos nulos antes
for col in onehot_cols + target_cols:
    X_train_log[col] = X_train_log[col].fillna('Missing')
    X_test_log[col] = X_test_log[col].fillna('Missing')

# Detectamos columnas numéricas automáticamente
num_cols = X_train_log.select_dtypes(include=['int64', 'float64']).columns

# Rellenamos con -1
X_train_log[num_cols] = X_train_log[num_cols].fillna(-1)
X_test_log[num_cols] = X_test_log[num_cols].fillna(-1)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression

# Preprocesado
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
        ('target', TargetEncoder(smooth="auto"), target_cols)
    ],
    remainder='passthrough'
)

pipeline_logreg = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_logreg.fit(X_train_log, y_train_log)

y_pred_logreg = pipeline_logreg.predict(X_test_log)
y_proba_logreg = pipeline_logreg.predict_proba(X_test_log)[:, 1]


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
print("Accuracy:", accuracy_score(y_test_log, y_pred_logreg))
print("ROC AUC:", roc_auc_score(y_test_log, y_proba_logreg))

Accuracy: 0.5304425216205598
ROC AUC: 0.5444185517153625


## Resumen de los resultados para los tres modelos

Algo a destacar es que los datos no siguen una relación lineal fuerte. Esto se ve en los resultados de los modelos:

- LightGBM

    - AUC: 0.7079
    - Accuracy: 0.6472
    - F1: 0.6404

- Random Forest

    - AUC: 0.7009
    - Accuracy: 0.6410

- Logistic regression
  - Accuracy: 0.52996390
  - ROC AUC: 0.54566
